# Aggregation Quality Analysis

This notebook demonstrates how to analyze the quality of time series aggregation using tsam's built-in plotting tools.

We will:
1. Load and aggregate time series data
2. Visualize the original vs reconstructed data
3. Analyze cluster structure and assignments
4. Examine residuals and error patterns
5. Compare different aggregation configurations
6. Analyze segmentation results

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, ExtremeConfig, SegmentConfig

pio.renderers.default = "notebook"

## 1. Load Data and Run Aggregation

In [ ]:
# Load test data (8760 hours = 1 year of hourly data)
raw = pd.read_csv("testdata.csv", index_col=0)
print(f"Data shape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
raw.head()

In [ ]:
# Run aggregation with 12 typical days
result = tsam.aggregate(
    raw,
    n_clusters=12,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
)

print(f"Number of clusters: {result.n_clusters}")
print(f"Timesteps per period: {result.n_timesteps_per_period}")
print(f"Total original periods: {len(raw) // result.n_timesteps_per_period}")

## 2. Visual Comparison: Original vs Reconstructed

### Heatmaps

Heatmaps show the full year with periods (days) on the x-axis and timesteps (hours) on the y-axis.

In [ ]:
# Standalone heatmap function on raw data
tsam.plot.heatmap(
    raw, column="T", period_duration=24, title="Original Temperature (standalone)"
)

In [ ]:
# Accessor: Original data heatmap
result.plot.heatmap(column="T", use_original=True, title="Original Temperature")

In [ ]:
# Accessor: Reconstructed data heatmap
result.plot.heatmap(column="T", title="Reconstructed Temperature")

In [ ]:
# Accessor: Multi-column heatmaps
result.plot.heatmaps(columns=["GHI", "T", "Load"], title="Reconstructed Time Series")

In [ ]:
# Standalone heatmaps with reference_data for consistent color scaling
# Use original data as reference so reconstructed colors match original scale
tsam.plot.heatmaps(
    result.reconstruct(),
    columns=["T", "Load"],
    period_duration=24,
    reference_data=raw,
    title="Reconstructed (color scale matched to original)",
)

### Duration Curves

Duration curves show sorted values and reveal how well the aggregation preserves the value distribution.

In [ ]:
# Standalone duration curve on raw data
tsam.plot.duration_curve(
    raw, columns=["Load", "GHI"], title="Original Duration Curves (standalone)"
)

In [ ]:
# Accessor: Compare original vs reconstructed duration curves
result.plot.compare(columns=["Load"], mode="duration_curve")

In [ ]:
# Accessor: Duration curves for reconstructed data
result.plot.duration_curve()

### Time Series Comparison

Zoom into specific time periods to see how well the aggregation captures temporal patterns.

In [ ]:
# Accessor: Compare overlay mode (same color per column, dash differentiates Original/Reconstructed)
result.plot.compare(
    columns=["T", "Load"],
    mode="overlay",
    start="20100115",
    end="20100122",
    title="Winter Week Comparison (overlay)",
)

In [ ]:
# Accessor: Compare side-by-side mode
result.plot.compare(
    columns=["GHI"],
    mode="side_by_side",
    start="20100701",
    end="20100708",
    title="Summer Week - Solar Irradiance (side_by_side)",
)

## 3. Cluster Analysis

Understanding the cluster structure helps assess whether the aggregation captures meaningful patterns.

In [ ]:
# Cluster weights - how many days are represented by each typical day
result.plot.cluster_weights()

In [ ]:
# Cluster assignments - which cluster each original day belongs to
result.plot.cluster_assignments()

In [ ]:
# Representative profiles for temperature
result.plot.cluster_representatives(columns=["T"])

In [ ]:
# Representative profiles for solar irradiance
result.plot.cluster_representatives(columns=["GHI"])

## 4. Error Analysis

### Accuracy Metrics

In [ ]:
# Overall accuracy metrics
print("Accuracy Summary:")
print(result.accuracy)
print("\nRMSE per column:")
print(result.accuracy.rmse)
print("\nMAE per column:")
print(result.accuracy.mae)

In [ ]:
# Visual comparison of accuracy metrics
result.plot.accuracy()

### Residual Analysis

Residuals (original - reconstructed) reveal where the aggregation performs well or poorly.

In [ ]:
# Residuals over time (mode="time_series")
result.plot.residuals(columns=["Load"], mode="time_series")

In [ ]:
# Residual distribution (mode="histogram")
result.plot.residuals(columns=["T", "Load"], mode="histogram")

In [ ]:
# Error by period (mode="by_period")
result.plot.residuals(columns=["Load"], mode="by_period")

In [ ]:
# Error by timestep within period (mode="by_timestep")
result.plot.residuals(columns=["Load", "GHI"], mode="by_timestep")

## 5. Comparing Aggregation Configurations

Compare different numbers of clusters to see the accuracy-complexity tradeoff.

In [ ]:
# Run aggregations with different cluster counts
results = {}
for n in [4, 8, 12, 24]:
    results[f"{n} clusters"] = tsam.aggregate(
        raw,
        n_clusters=n,
        period_duration=24,
        cluster=ClusterConfig(method="hierarchical"),
    )

# Print accuracy comparison
print("RMSE comparison (Load):")
for name, res in results.items():
    print(f"  {name}: {res.accuracy.rmse['Load']:.2f}")

In [ ]:
# Standalone compare: duration curves across configurations
comparison_data = {"Original": raw}
for name, res in results.items():
    comparison_data[name] = res.reconstruct()

tsam.plot.compare(
    comparison_data,
    column="Load",
    plot_type="duration_curve",
    title="Duration Curve: Cluster Count Comparison",
)

In [ ]:
# Standalone compare: time slice across configurations
tsam.plot.compare(
    comparison_data,
    column="Load",
    plot_type="time_slice",
    start="20100601",
    end="20100608",
    title="June Week: Cluster Count Comparison",
)

## 6. Effect of Extreme Period Preservation

Compare aggregation with and without preserving extreme values.

In [ ]:
# Without extreme preservation
result_no_extremes = tsam.aggregate(
    raw,
    n_clusters=8,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
)

# With extreme preservation
result_with_extremes = tsam.aggregate(
    raw,
    n_clusters=8,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
    extremes=ExtremeConfig(
        method="new_cluster",
        min_value=["T"],
        max_value=["Load", "GHI"],
    ),
)

print("Without extremes - Load RMSE:", result_no_extremes.accuracy.rmse["Load"])
print("With extremes - Load RMSE:", result_with_extremes.accuracy.rmse["Load"])

In [ ]:
# Compare peak preservation in duration curves
tsam.plot.compare(
    {
        "Original": raw,
        "No extremes": result_no_extremes.reconstruct(),
        "With extremes": result_with_extremes.reconstruct(),
    },
    column="Load",
    plot_type="duration_curve",
    title="Effect of Extreme Period Preservation on Load",
)

In [ ]:
# Compare temperature extremes
tsam.plot.compare(
    {
        "Original": raw,
        "No extremes": result_no_extremes.reconstruct(),
        "With extremes": result_with_extremes.reconstruct(),
    },
    column="T",
    plot_type="duration_curve",
    title="Effect of Extreme Period Preservation on Temperature",
)

## 7. Segmentation Analysis

When using segmentation, you can visualize the segment durations.

In [ ]:
# Run aggregation with segmentation
result_segmented = tsam.aggregate(
    raw,
    n_clusters=12,
    period_duration=24,
    cluster=ClusterConfig(method="hierarchical"),
    segments=SegmentConfig(n_segments=6),
)

print(f"Segments per period: {len(result_segmented.segment_durations[0])}")
print(f"Segment durations (first cluster): {result_segmented.segment_durations[0]}")

In [ ]:
# Plot segment durations
result_segmented.plot.segment_durations()

In [ ]:
# Compare segmented vs non-segmented
tsam.plot.compare(
    {
        "Original": raw,
        "No segmentation": result.reconstruct(),
        "With segmentation": result_segmented.reconstruct(),
    },
    column="Load",
    plot_type="duration_curve",
    title="Effect of Segmentation on Load Duration Curve",
)

## Summary

### Plotting Functions Overview

**Standalone functions (`tsam.plot.*`):**
- `heatmap(data, column, ...)` - Single column heatmap
- `heatmaps(data, columns, reference_data, ...)` - Multi-column stacked heatmaps with optional reference scaling
- `duration_curve(data, columns, ...)` - Sorted value curves
- `compare(results_dict, column, plot_type, ...)` - Compare multiple DataFrames
  - `plot_type="duration_curve"` - Sorted value comparison
  - `plot_type="time_slice"` - Time window comparison

**Accessor methods (`result.plot.*`):**
- `heatmap(column, use_original, ...)` - Heatmap of reconstructed/original
- `heatmaps(columns, ...)` - Multi-column heatmaps
- `duration_curve(columns, ...)` - Duration curves of reconstructed data
- `compare(columns, mode, ...)` - Compare original vs reconstructed
  - `mode="overlay"` - Same plot, color=column, dash=source
  - `mode="side_by_side"` - Faceted by source
  - `mode="duration_curve"` - Sorted value comparison
- `cluster_weights()` - Bar chart of cluster sizes
- `cluster_assignments()` - Heatmap of period-to-cluster mapping
- `cluster_representatives(columns)` - Line plots of typical periods
- `accuracy()` - Bar chart of RMSE/MAE metrics
- `segment_durations()` - Bar chart of segment lengths (requires segmentation)
- `residuals(columns, mode, ...)` - Error analysis
  - `mode="time_series"` - Residuals over time
  - `mode="histogram"` - Error distribution
  - `mode="by_period"` - MAE per original period
  - `mode="by_timestep"` - MAE by hour within period